# PrimeKV — Weekend Notebook

One notebook that tests the three things we could not validate on Day 1:

1. **SpaCy classifier** — does a linguistically-aware classifier beat the rule-based one? (gating experiment)
2. **Dynamic tuning** — do policies auto-adapt as memory pressure changes?
3. **Reasoning eval** — does PrimeKV preserve constraints through filler that H2O / StreamingLLM drop?

Target runtime: Colab A100 or L4. Qwen2.5-3B-Instruct on GPU.

Run top-to-bottom.

## 1. Install

Clones the repo, installs the package, downloads the spaCy English model.

In [ ]:
!rm -rf /content/PrimeKV
!git clone --branch primekv2 https://github.com/arunvenkatadri/PrimeKV.git /content/PrimeKV
%cd /content/PrimeKV
!pip install -q -e .[spacy,web]
!python -m spacy download en_core_web_sm

## 2. Load Qwen2.5-3B-Instruct on GPU

Requires `attn_implementation="eager"` so `past_key_values` stays in the standard tuple layout the PrimeKV adapter expects.

In [ ]:
%matplotlib inline
import torch
assert torch.cuda.is_available(), 'Change runtime type to GPU (A100/L4 recommended)'
print('GPU:', torch.cuda.get_device_name(0))

from transformers import AutoModelForCausalLM, AutoTokenizer
MODEL_NAME = 'Qwen/Qwen2.5-3B-Instruct'
tok = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)
if tok.pad_token is None:
    tok.pad_token = tok.eos_token
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,
    attn_implementation='eager',
).to('cuda').eval()
print('loaded', MODEL_NAME, '—', model.config.num_hidden_layers, 'layers,',
      model.config.num_attention_heads, 'heads,',
      model.config.hidden_size // model.config.num_attention_heads, 'head_dim')

# --- Publication plotting style ---
import os
import matplotlib.pyplot as plt

FIG_DIR = '/content/PrimeKV/figures'
os.makedirs(FIG_DIR, exist_ok=True)

plt.rcParams.update({
    'figure.dpi': 110,
    'savefig.dpi': 300,
    'savefig.bbox': 'tight',
    'savefig.facecolor': 'white',
    'font.family': 'DejaVu Sans',
    'font.size': 11,
    'axes.titlesize': 13,
    'axes.titleweight': 'bold',
    'axes.labelsize': 11,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'axes.grid': True,
    'grid.alpha': 0.25,
    'grid.linestyle': '--',
    'legend.frameon': False,
    'legend.fontsize': 10,
    'xtick.labelsize': 10,
    'ytick.labelsize': 10,
})

CACHE_COLORS = {
    'full':          '#2d2d2d',
    'primekv_spacy': '#e05a00',
    'primekv_rule':  '#f5b041',
    'h2o':           '#1f77b4',
    'streaming':     '#2ca02c',
    'uniform_int4':  '#d62728',
    'uniform_int8':  '#8c564b',
}
TIER_COLORS = {'ANCHOR': '#2d2d2d', 'SEMANTIC': '#e05a00', 'SUPPORTING': '#1f77b4', 'FILLER': '#d62728'}

def save_fig(fig, name):
    out = f'{FIG_DIR}/{name}.png'
    fig.savefig(out, dpi=300, bbox_inches='tight')
    print(f'  saved -> {out}')
    return out

# A realistic long-context prompt for all downstream experiments.
LONG_PROMPT = (
    'The Eiffel Tower is a wrought-iron lattice tower on the Champ de Mars in Paris, France. '
    'It is named after the engineer Gustave Eiffel, whose company designed and built the tower '
    'from 1887 to 1889. Locally nicknamed La dame de fer, it was constructed as the centerpiece '
    'of the 1889 Worlds Fair, and to crown the 100th anniversary of the French Revolution. '
    'Although initially criticised by some of Frances leading artists and intellectuals for its '
    'design, it has since become a global cultural icon of France and one of the most recognisable '
    'structures in the world. '
    'The tower is 330 metres tall, about the same height as an 81-storey building, and was the '
    'tallest man-made structure in the world for 41 years until the Chrysler Building in New York '
    'City was finished in 1930. It was the first structure in the world to surpass both the 200-metre '
    'and 300-metre marks. Due to the addition of a broadcasting aerial at the top of the tower in '
    '1957, it is now taller than the Chrysler Building by 5.2 metres. Excluding transmitters, the '
    'Eiffel Tower is the second tallest free-standing structure in France after the Millau Viaduct. '
) * 3
print('prompt length (chars):', len(LONG_PROMPT))
print('prompt length (tokens):', len(tok(LONG_PROMPT)['input_ids']))

## 3. SpaCy classifier sweep — the gating experiment

Does a classifier that uses actual POS tags / named entities beat a positional stride? If rule-based wins here, the structural-role thesis is not validated yet.

We compare three PrimeKV variants (rule-based vs spaCy) against the full cache and two baselines at matched memory.

In [ ]:
import spacy
import numpy as np
import matplotlib.pyplot as plt
from primekv.cache import PrimeKVCache, TierPolicy, DEFAULT_POLICIES
from primekv.classifier import RuleBasedClassifier, SpaCyClassifier, Tier
from primekv.baselines import FullCache, H2OCache, StreamingLLMCache, UniformQuantCache
from primekv.eval import Workload, run_comparison

nlp = spacy.load('en_core_web_sm')
NUM_LAYERS = model.config.num_hidden_layers

def make_primekv(classifier):
    return PrimeKVCache(
        num_layers=NUM_LAYERS,
        classifier=classifier,
        policies=dict(DEFAULT_POLICIES),
        device='cuda',
    )

# --- Figure A: tier distribution, rule-based vs spaCy ---
# This is the qualitative hypothesis test: does spaCy actually put
# DIFFERENT tokens in SEMANTIC than rule-based does?
rule_clf = RuleBasedClassifier(anchor_prefix_len=4, semantic_stride=3)
spacy_clf = SpaCyClassifier(nlp=nlp, anchor_prefix_len=4)
ids = tok(LONG_PROMPT, return_tensors='pt')['input_ids'][0]
rule_out = rule_clf.classify(input_ids=ids)
spacy_out = spacy_clf.classify_text(LONG_PROMPT, tok)

tier_order = [Tier.ANCHOR, Tier.SEMANTIC, Tier.SUPPORTING, Tier.FILLER]
rule_counts = [int((rule_out.tiers == int(t)).sum()) for t in tier_order]
spacy_counts = [int((spacy_out.tiers == int(t)).sum()) for t in tier_order]

fig, ax = plt.subplots(figsize=(9, 4.2))
x = np.arange(2)
bottom_r, bottom_s = 0, 0
for t, rc, sc in zip(tier_order, rule_counts, spacy_counts):
    color = TIER_COLORS[t.name]
    ax.bar(0, rc, bottom=bottom_r, color=color, label=t.name, width=0.55)
    ax.bar(1, sc, bottom=bottom_s, color=color, width=0.55)
    bottom_r += rc; bottom_s += sc
ax.set_xticks([0, 1]); ax.set_xticklabels(['rule-based', 'spaCy POS/NER'])
ax.set_ylabel('tokens')
ax.set_title('Tier assignment: same prompt, different classifiers')
ax.legend(loc='upper right', ncol=4, bbox_to_anchor=(1.0, 1.15))
plt.tight_layout()
save_fig(fig, 'fig1_tier_distribution')
plt.show()

# --- Figure B: Pareto scatter — perplexity vs memory across all caches ---
caches = {
    'full':          FullCache(num_layers=NUM_LAYERS),
    'primekv_rule':  make_primekv(RuleBasedClassifier(anchor_prefix_len=4, semantic_stride=3)),
    'primekv_spacy': make_primekv(SpaCyClassifier(nlp=nlp, anchor_prefix_len=4)),
    'h2o':           H2OCache(num_layers=NUM_LAYERS, capacity=256),
    'streaming':     StreamingLLMCache(num_layers=NUM_LAYERS, num_sinks=4, window=256),
    'uniform_int4':  UniformQuantCache(num_layers=NUM_LAYERS, bits=4),
}
workload = Workload(prompt=LONG_PROMPT, decode_tokens=32, max_length=3072, name='spacy_gate')
report = run_comparison(caches, workload, model, tok, device='cuda')
print(report.to_markdown())

fig, ax = plt.subplots(figsize=(9, 5.5))
for r in report.results:
    color = CACHE_COLORS.get(r.name, '#888')
    mem_mb = r.memory_bytes / (1024*1024)
    ax.scatter(mem_mb, r.perplexity, s=160, color=color,
               edgecolor='white', linewidth=1.5, zorder=3, label=r.name)
    ax.annotate(r.name, (mem_mb, r.perplexity),
                xytext=(8, 6), textcoords='offset points', fontsize=9)
ax.set_xlabel('cache memory (MB) — lower is better')
ax.set_ylabel('perplexity — lower is better')
ax.set_title('Quality vs memory: does spaCy classification help PrimeKV?')
ax.set_xscale('log')
plt.tight_layout()
save_fig(fig, 'fig2_spacy_gate_pareto')
plt.show()

## 4. Dynamic tuning sweep

Vary the memory budget; watch `auto_tune` pick different operating points and `build_cache_from_profile` materialize each. The perplexity of each profile tells us how well the method degrades under pressure.

In [ ]:
from primekv.tuning import auto_tune, build_cache_from_profile, describe_profile
import numpy as np
import matplotlib.pyplot as plt

prompt_len = len(tok(LONG_PROMPT)['input_ids'])
budgets = [None, 128.0, 32.0, 8.0, 2.0]

tuning_caches = {'full': FullCache(num_layers=NUM_LAYERS)}
profiles = {}
for b in budgets:
    profile = auto_tune(
        prompt_length=prompt_len,
        memory_budget_mb=b,
        num_layers=NUM_LAYERS,
        num_heads=model.config.num_attention_heads,
        head_dim=model.config.hidden_size // model.config.num_attention_heads,
    )
    label = f'{"no-budget" if b is None else str(int(b)) + "MB"}_[{profile.name}]'
    profiles[label] = profile
    tuning_caches[label] = build_cache_from_profile(profile, num_layers=NUM_LAYERS, device='cuda')

for label, p in profiles.items():
    print(f'=== {label} ===')
    print(describe_profile(p))
    print()

tuning_report = run_comparison(tuning_caches, workload, model, tok, device='cuda')
print(tuning_report.to_markdown())

# Publication figure: two panels — (left) perplexity per profile, (right)
# measured memory vs profile's estimated memory.
names = [r.name for r in tuning_report.results]
ppls = [r.perplexity for r in tuning_report.results]
meas_mem = [r.memory_bytes / (1024*1024) for r in tuning_report.results]
est_mem = [None if n == 'full' else profiles[n].estimated_bytes / (1024*1024) for n in names]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4.5))
xs = np.arange(len(names))

bars = ax1.bar(xs, ppls, color='#e05a00', alpha=0.85, edgecolor='white')
ax1.set_xticks(xs); ax1.set_xticklabels(names, rotation=25, ha='right', fontsize=9)
ax1.set_ylabel('perplexity')
ax1.set_title('Quality holds as budget tightens')
for bar, v in zip(bars, ppls):
    ax1.text(bar.get_x()+bar.get_width()/2, v, f'{v:.2f}', ha='center', va='bottom', fontsize=9)

ax2.bar(xs - 0.18, meas_mem, width=0.36, label='measured', color='#1f77b4', edgecolor='white')
ax2.bar(xs + 0.18, [m if m is not None else 0 for m in est_mem], width=0.36,
         label='auto_tune estimate', color='#ff7f0e', alpha=0.7, edgecolor='white')
ax2.set_xticks(xs); ax2.set_xticklabels(names, rotation=25, ha='right', fontsize=9)
ax2.set_ylabel('cache memory (MB)')
ax2.set_title('auto_tune estimate tracks measured memory')
ax2.legend()
plt.tight_layout()
save_fig(fig, 'fig3_dynamic_tuning')
plt.show()

## 5. Reasoning eval — constraint persistence

This is the test perplexity can't see: does the cache preserve a rule ("never mention blue") across a long filler section? Pass/fail on each of 4 canonical tests.

In [ ]:
from primekv.reasoning_eval import default_tests, run_reasoning_eval
import numpy as np
import matplotlib.pyplot as plt

reasoning_caches = {
    'full':          FullCache(num_layers=NUM_LAYERS),
    'primekv_spacy': make_primekv(SpaCyClassifier(nlp=nlp, anchor_prefix_len=4)),
    'primekv_rule':  make_primekv(RuleBasedClassifier(anchor_prefix_len=4, semantic_stride=3)),
    'h2o':           H2OCache(num_layers=NUM_LAYERS, capacity=128),
    'streaming':     StreamingLLMCache(num_layers=NUM_LAYERS, num_sinks=4, window=128),
}

reasoning_report = run_reasoning_eval(
    reasoning_caches, model, tok, tests=default_tests(), device='cuda', max_length=3072,
)
print(reasoning_report.to_markdown())

# Figure: pass-rate bars (left) + per-test heatmap (right).
rates = reasoning_report.pass_rate_per_cache()
cache_names = list(rates.keys())
test_names = []
seen = set()
for r in reasoning_report.results:
    if r.test not in seen:
        test_names.append(r.test); seen.add(r.test)

grid = np.zeros((len(test_names), len(cache_names)))
for i, t in enumerate(test_names):
    for j, c in enumerate(cache_names):
        res = next((r for r in reasoning_report.results if r.test == t and r.cache == c), None)
        grid[i, j] = 1.0 if (res and res.passed) else 0.0

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4.5), gridspec_kw={'width_ratios': [1, 1.3]})

colors = [CACHE_COLORS.get(n, '#888') for n in cache_names]
bars = ax1.bar(cache_names, [rates[n] for n in cache_names], color=colors, edgecolor='white')
ax1.set_ylim(0, 1.05)
ax1.set_ylabel('pass rate')
ax1.set_title('Constraint persistence: overall pass rate')
for bar, n in zip(bars, cache_names):
    ax1.text(bar.get_x()+bar.get_width()/2, rates[n]+0.03, f'{rates[n]:.0%}',
             ha='center', va='bottom', fontweight='bold')
ax1.tick_params(axis='x', rotation=25)

im = ax2.imshow(grid, cmap='RdYlGn', vmin=0, vmax=1, aspect='auto')
ax2.set_xticks(range(len(cache_names))); ax2.set_xticklabels(cache_names, rotation=25, ha='right')
ax2.set_yticks(range(len(test_names))); ax2.set_yticklabels(test_names)
ax2.set_title('Per-test outcome (green = pass, red = fail)')
ax2.grid(False)
for i in range(len(test_names)):
    for j in range(len(cache_names)):
        ax2.text(j, i, '✓' if grid[i, j] else '✗',
                 ha='center', va='center', color='white', fontsize=14, fontweight='bold')
plt.tight_layout()
save_fig(fig, 'fig4_reasoning_eval')
plt.show()

In [ ]:
# Per-test, per-cache debug view — useful for figuring out *why* a cache failed.
for r in reasoning_report.results:
    tail = (r.generated or '').rsplit('Answer:', 1)[-1].strip().replace('\n', ' ')[:100]
    print(f"[{r.cache:15s}] {r.test:20s} {'PASS' if r.passed else 'FAIL'}  ({r.reason})")
    print(f"{'':19s}  → {tail}")

## 6. Headline figure — 2D Pareto with SpaCy classifier

The 2D eviction × quantization sweep from Day 1, re-run with the SpaCy classifier instead of rule-based. If SpaCy shifts the PrimeKV curve left (lower PPL at same compression) vs rule-based, the thesis has support.

In [ ]:
from primekv.sweep import sweep_2d_tradeoff
import numpy as np
import matplotlib.pyplot as plt

caps = [32, 64, 128, 256, 512]
precisions = ['fp16', 'int8', 'int4']

def spacy_classifier_factory():
    return SpaCyClassifier(nlp=nlp, anchor_prefix_len=4)

report_2d = sweep_2d_tradeoff(
    model=model, tokenizer=tok, prompt=LONG_PROMPT,
    eviction_caps=caps, precisions=precisions,
    decode_tokens=16, max_length=2048, device='cuda',
    primekv_classifier_factory=spacy_classifier_factory,
)

# Build our own publication plot rather than using plot_report so we
# can annotate the PrimeKV operating point and match the notebook style.
fig, ax = plt.subplots(figsize=(10, 6))
groups = report_2d.by_cache()

MARKERS = {'fp16': 'o', 'int8': 's', 'int4': '^'}

for name, pts in groups.items():
    color = CACHE_COLORS.get(name, '#888')
    pts_with_ppl = [p for p in pts if p.perplexity is not None]
    if not pts_with_ppl:
        continue
    if name == 'primekv':
        by_prec = {}
        for p in pts_with_ppl:
            prec = p.extra.get('precision', 'fp16')
            by_prec.setdefault(prec, []).append(p)
        for prec, prec_pts in by_prec.items():
            prec_pts.sort(key=lambda p: p.compression_ratio)
            xs = [p.compression_ratio for p in prec_pts]
            ys = [p.perplexity for p in prec_pts]
            ax.plot(xs, ys, marker=MARKERS.get(prec, 'o'), markersize=9,
                    linewidth=2.0, color=color, alpha=0.95,
                    label=f'primekv (Tier-2 {prec})', markeredgecolor='white')
    else:
        pts_with_ppl.sort(key=lambda p: p.compression_ratio)
        xs = [p.compression_ratio for p in pts_with_ppl]
        ys = [p.perplexity for p in pts_with_ppl]
        if len(pts_with_ppl) == 1:
            ax.scatter(xs, ys, s=180, color=color, label=name,
                       edgecolor='white', linewidth=1.5, zorder=3)
        else:
            ax.plot(xs, ys, marker='o', markersize=9, linewidth=2.0,
                    color=color, label=name, markeredgecolor='white')

ax.set_xscale('log')
ax.set_xlabel('compression ratio (higher = more compressed)')
ax.set_ylabel('perplexity (lower = better)')
ax.set_title('PrimeKV two-lever control surface vs single-lever baselines\n'
             '(spaCy classifier, Qwen2.5-3B-Instruct)')
ax.legend(loc='best', ncol=2)
plt.tight_layout()
save_fig(fig, 'fig5_headline_2d_pareto')
plt.show()

## 7. (Optional) Autoresearch — Karpathy-style overnight loop

Requires `ANTHROPIC_API_KEY` in the env. Runs `--rounds N` of read-propose-evaluate-keep against `autoresearch/experiment.py`. The evaluator uses tiny-gpt2 for speed, so one round is ~30s.

Skip this cell if you don't have an API key.

In [ ]:
%cd /content/PrimeKV
import os
if not os.environ.get('ANTHROPIC_API_KEY'):
    print('Set ANTHROPIC_API_KEY to run this cell.')
else:
    !pip install -q anthropic
    !python -m autoresearch.run --agent anthropic --rounds 3 --device cpu

## Takeaways & publication artifacts

After the cells have run, you have five publication-ready PNGs in `/content/PrimeKV/figures/` at 300 DPI:

| file | figure | what it shows |
|---|---|---|
| `fig1_tier_distribution.png` | Fig 1 | Qualitative: rule-based vs spaCy assign tokens to different tiers. |
| `fig2_spacy_gate_pareto.png` | Fig 2 | Quality vs memory scatter across 6 caches. Is spaCy left and/or low of rule-based? |
| `fig3_dynamic_tuning.png` | Fig 3 | Perplexity + memory across memory budgets; auto_tune estimate vs measured. |
| `fig4_reasoning_eval.png` | Fig 4 | Pass rates + per-test heatmap. The constraint-persistence story. |
| `fig5_headline_2d_pareto.png` | Fig 5 | The two-lever control surface — PrimeKV eviction × quantization. |

Download them from the Colab file browser or run `!cp /content/PrimeKV/figures/*.png /content/drive/MyDrive/` after mounting Drive.

**Honest observations to fill in:**

- **SpaCy vs rule-based (§3, Fig 1–2):** ___
- **Tuning adapts under pressure (§4, Fig 3):** ___
- **Reasoning pass rates (§5, Fig 4):** ___
- **2D Pareto movement (§6, Fig 5):** ___